In [1]:
import torch, gc
import random

from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from model_utils import load_transformer_model, load_silence_latent, load_encoder, decode_latent_and_save_audio, \
    get_files_in_path_as_array, load_finetuning_audio_segments, encode_audio_segments

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- config ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model_dtype = torch.bfloat16
model_repo = "ACE-Step/acestep-v15-turbo-shift1"
checkpoint_dir = './checkpoints/'
checkpoint_file = checkpoint_dir + 'checkpoint.pt'

cuda


In [ ]:
# encode training audio to VAE latent space in batches (VRAM limited)
vae = load_encoder("./models/ace-step-vae/config.json", "./models/ace-step-vae/checkpoint.ckpt", device, model_dtype)
path = "inputs/lora"
load_batch_size = 2
train_data_limit = 30
files = get_files_in_path_as_array(path)
wavs = load_finetuning_audio_segments(files, device, model_dtype)
y = torch.Tensor().to(device).to(model_dtype)
total = min(len(wavs), train_data_limit)
with torch.no_grad():
    for start in tqdm(range(0, total, load_batch_size)):
        end = min(start + load_batch_size, total)
        batch = encode_audio_segments(vae, wavs[start:end], device, model_dtype)
        y = torch.cat((y, batch), 0)
        del batch
        torch.cuda.empty_cache()
        gc.collect()
del vae

In [ ]:
# 80/20 train/test split, shuffled
class AudioLatentDataset(Dataset):
    def __init__(self, audio_latents):
        self.audio_latents = audio_latents

    def __len__(self):
        return self.audio_latents.shape[0]

    def __getitem__(self, item):
        return self.audio_latents[item]

n = y.shape[0]
perm = torch.randperm(n)
y_shuffled = y[perm]

split = int(0.8 * n)
train_dataset = AudioLatentDataset(y_shuffled[:split])
test_dataset = AudioLatentDataset(y_shuffled[split:])

In [5]:
batch_size = 1
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=True)

In [3]:
dit = load_transformer_model(model_repo, model_dtype, device)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:454: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:639: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\finite_scalar_quantization.py:159: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\lookup_free_quantization.py:244: FutureWarning: `torch.cuda.amp.autocast(args...)

In [4]:
# freeze all base model params before adding LoRA
for parameter in dit.parameters():
    parameter.requires_grad = False

In [5]:
import math

# LoRA: low-rank adaptation. A and B are the low-rank matrices (dW = B @ A).
# rank controls capacity, alpha/rank = scaling factor applied to dW.
rank = 8
lora_alpha = rank

def init_lora_params(proj, name, rank, device, dtype):
    # A gets small random init, B gets zeros (so dW starts at zero)
    setattr(proj, f"{name}_A", nn.Parameter(
        torch.randn(rank, proj.in_features, device=device, dtype=dtype) / math.sqrt(rank)))
    setattr(proj, f"{name}_B", nn.Parameter(
        torch.zeros(proj.out_features, rank, device=device, dtype=dtype)))

# only train the second half of decoder layers (earlier layers are more general)
train_from_layer_idx = int(len(dit.decoder.layers) / 2)
for layer in dit.decoder.layers[train_from_layer_idx:]:
    for name, proj in [("q", layer.self_attn.q_proj), ("k", layer.self_attn.k_proj),
                       ("v", layer.self_attn.v_proj), ("o", layer.self_attn.o_proj)]:
        init_lora_params(proj, name, rank, device, model_dtype)

In [6]:
# lora_scale = alpha / rank, controls how much the low-rank update affects output
lora_scale = lora_alpha / rank

def make_lora_hook(name):
    # hook adds the LoRA delta: output += scale * x @ (B @ A).T
    def hook(module, inputs, outputs):
        x = inputs[0]
        A = getattr(module, f"{name}_A")
        B = getattr(module, f"{name}_B")
        return outputs + lora_scale * torch.matmul(x, torch.matmul(B, A).T)
    return hook

hooks = []
for layer in dit.decoder.layers[train_from_layer_idx:]:
    for name, proj in [("q", layer.self_attn.q_proj), ("k", layer.self_attn.k_proj),
                       ("v", layer.self_attn.v_proj), ("o", layer.self_attn.o_proj)]:
        hooks.append(proj.register_forward_hook(make_lora_hook(name)))

In [7]:
# OPTIONAL: Run this to load checkpoint
dit.decoder.load_state_dict(torch.load(checkpoint_file, weights_only=True))

<All keys matched successfully>

In [8]:
silence_latent = load_silence_latent(model_repo, "silence_latent.pt", device, model_dtype)

In [9]:
def build_empty_conditions(silence_latent, batch_size, device, model_dtype, seconds=60):
    # zero-valued text/lyric embeddings + silence reference = unconditional generation
    fps = 25
    seq_len = int(seconds * fps)
    text_hidden_states = torch.zeros(batch_size, 77, 1024, dtype=model_dtype, device=device)
    text_attention_mask = torch.zeros(batch_size, 77, dtype=torch.bool, device=device)
    lyric_hidden_states = torch.zeros(batch_size, 123, 1024, dtype=model_dtype, device=device)
    lyric_attention_mask = torch.zeros(batch_size, 123, dtype=torch.bool, device=device)
    is_covers = torch.Tensor([False]).repeat(batch_size).to(device)
    refer_audio = silence_latent[:, :, :1500].repeat(batch_size, 1, 1).permute(0, 2, 1)
    refer_mask = torch.LongTensor(range(0, batch_size)).to(device)
    chunk_mask = torch.ones(batch_size, seq_len, 64, dtype=torch.bool, device=device)
    src_latents = silence_latent[:, :, :seq_len].repeat(batch_size, 1, 1).permute(0, 2, 1)
    return {
        "text_hidden_states": text_hidden_states, "text_attention_mask": text_attention_mask,
        "lyric_hidden_states": lyric_hidden_states, "lyric_attention_mask": lyric_attention_mask,
        "is_covers": is_covers, "refer_audio": refer_audio, "refer_mask": refer_mask,
        "chunk_mask": chunk_mask, "src_latents": src_latents, "seq_len": seq_len,
    }

def do_inference(model, batch_size=1):
    c = build_empty_conditions(silence_latent, batch_size, device, model_dtype)

    outputs = model.generate_audio(
        text_hidden_states=c["text_hidden_states"],
        text_attention_mask=c["text_attention_mask"],
        lyric_hidden_states=c["lyric_hidden_states"],
        lyric_attention_mask=c["lyric_attention_mask"],
        refer_audio_acoustic_hidden_states_packed=c["refer_audio"],
        refer_audio_order_mask=c["refer_mask"],
        src_latents=c["src_latents"],
        chunk_masks=c["chunk_mask"],
        infer_steps=50,
        is_covers=c["is_covers"],
        silence_latent=silence_latent,
        use_progress_bar=True,
        shift=1.0,
        repainting_start=torch.tensor([1.0]),
        repainting_end=torch.tensor([0.0]),
        audio_cover_strength=1.0,
        use_repainting=False
    )
    output_latents = outputs['target_latents'].transpose(1, 2).contiguous()
    return output_latents

In [10]:
# single denoising step through the decoder
def do_prediction(model, xt, t_curr_tensor, params):
    decoder_outputs = model.decoder(
        hidden_states=xt,
        timestep=t_curr_tensor,
        timestep_r=t_curr_tensor,
        attention_mask=params['attention_mask'],
        encoder_hidden_states=params['encoder_hidden_states'],
        encoder_attention_mask=params['encoder_attention_mask'],
        context_latents=params['context_latents'],
        use_cache=True,
        past_key_values=None,
    )
    return decoder_outputs[0]

In [11]:
# prepare conditioning tensors and noise for the training forward pass
def prepare_inference(model, silence_latent, batch_size, device, model_dtype):
    with torch.no_grad():
        c = build_empty_conditions(silence_latent, batch_size, device, model_dtype)
        attention_mask = torch.ones(batch_size, c["seq_len"], device=device, dtype=model_dtype)

        encoder_hidden_states, encoder_attention_mask, context_latents = model.prepare_condition(
            text_hidden_states=c["text_hidden_states"],
            text_attention_mask=c["text_attention_mask"],
            lyric_hidden_states=c["lyric_hidden_states"],
            lyric_attention_mask=c["lyric_attention_mask"],
            refer_audio_acoustic_hidden_states_packed=c["refer_audio"],
            refer_audio_order_mask=c["refer_mask"],
            hidden_states=c["src_latents"],
            attention_mask=attention_mask,
            silence_latent=silence_latent,
            src_latents=c["src_latents"],
            chunk_masks=c["chunk_mask"],
            is_covers=c["is_covers"],
            precomputed_lm_hints_25Hz=None,
            audio_codes=None,
        )

        noise = model.prepare_noise(context_latents, None)

    return {'attention_mask': attention_mask, 'encoder_hidden_states': encoder_hidden_states,
            'encoder_attention_mask': encoder_attention_mask, 'context_latents': context_latents, 'noise': noise}

In [12]:
# flow-matching timesteps: 8 evenly spaced from pure noise (1.0) to mostly clean (0.125)
TIMESTEPS = [1.0, 0.875, 0.75, 0.625, 0.5, 0.375, 0.25, 0.125]

def forward_pass(y_batch, model, batch_size):
    # sample random timestep and blend noise with clean latents
    t = TIMESTEPS[random.randrange(len(TIMESTEPS))]
    t_curr = t * torch.ones((batch_size,), device=device, dtype=model_dtype)

    params = prepare_inference(model, silence_latent, batch_size, device, model_dtype)
    noise = params["noise"]

    # interpolate: xt = t*noise + (1-t)*clean (flow matching formulation)
    t_ = t_curr.unsqueeze(-1).unsqueeze(-1)
    xt = noise * t_ + (1 - t_) * y_batch.permute(0, 2, 1)

    pred = do_prediction(model, xt, t_curr, params)
    # velocity target: v = noise - clean (model learns the flow field)
    target = noise - y_batch.permute(0, 2, 1)
    return pred, target

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    for batch, y in enumerate(dataloader):
        pred, target = forward_pass(y, model, batch_size)

        loss = loss_fn(pred, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in dit.parameters() if p.requires_grad], max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()

        loss, current = loss.item(), batch * batch_size
        layer = dit.decoder.layers[train_from_layer_idx]
        norm = torch.matmul(layer.self_attn.q_proj.q_B, layer.self_attn.q_proj.q_A).norm().item()
        print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]  q_dW_norm: {norm:.4f}")

def test_loop(dataloader, model, loss_fn):
    num_batches = len(dataloader)
    test_loss = 0

    with torch.no_grad():
        for y in dataloader:
            pred, target = forward_pass(y, model, batch_size)
            test_loss += loss_fn(pred, target).item()

    test_loss /= num_batches
    print(f"Avg Test loss: {test_loss:>8f} \n")

In [13]:
# reload VAE to decode latents back to audio (freed earlier for VRAM)
def decode_and_save(output_latents):
    vae = load_encoder("./models/ace-step-vae/config.json", "./models/ace-step-vae/checkpoint.ckpt", device,
                       model_dtype)
    with torch.no_grad():
        decode_latent_and_save_audio(output_latents, vae, "lora")
    del vae
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
# MSE loss on velocity prediction, AdamW with gradient clipping
loss_fn = nn.MSELoss()
lora_params = [p for p in dit.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(lora_params, lr=1e-3, weight_decay=0.01)

epochs = 30
for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train_loop(train_dataloader, dit, loss_fn, optimizer)
    test_loop(test_dataloader, dit, loss_fn)

In [17]:
torch.save(dit.decoder.state_dict(), checkpoint_file)

In [14]:
if hooks:
    for hook in hooks:
        hook.remove()

In [15]:
# merge LoRA weights into base model: W += scale * B @ A
with torch.no_grad():
    for layer in dit.decoder.layers[train_from_layer_idx:]:
        for name, proj in [("q", layer.self_attn.q_proj), ("k", layer.self_attn.k_proj),
                           ("v", layer.self_attn.v_proj), ("o", layer.self_attn.o_proj)]:
            A = getattr(proj, f"{name}_A")
            B = getattr(proj, f"{name}_B")
            proj.weight.add_(lora_scale * torch.matmul(B, A))

In [16]:
output_latents = do_inference(dit, 1)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\residual_fsq.py:170: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled = False):


In [17]:
del dit
torch.cuda.empty_cache()
gc.collect()

1543

In [18]:
decode_and_save(output_latents)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\clip\clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention


W0610 18:16:57.131000 22000 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


decoded shape: torch.Size([1, 2, 2880000])
Audio saved to outputs\lora0.wav
saved: (2880000, 2)
